In [ ]:
import IPython.display
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
import astropy.visualization
import named_arrays as na
import ctis

In [ ]:
wavelength = na.linspace(-500, 500, axis="wavelength", num=21) * u.km / u.s

In [ ]:
position_scene = na.Cartesian2dVectorLinearSpace(
    start=-10 * u.arcsec,
    stop=10 * u.arcsec,
    axis=na.Cartesian2dVectorArray("scene_x", "scene_y"),
    num=64,
)

In [ ]:
position_sensor = na.Cartesian2dVectorArray(
    x=na.arange(0, 64, axis="sensor_x") * u.pix,
    y=na.arange(0, 64, axis="sensor_y") * u.pix,
)

In [ ]:
coordinates_scene = na.SpectralPositionalVectorArray(wavelength, position_scene)
coordinates_sensor = na.SpectralPositionalVectorArray(wavelength, position_sensor)

In [ ]:
instrument = ctis.instruments.IdealInstrument(
    response=1,
    plate_scale=.4 * u.arcsec / u.pix,
    # dispersion=na.ScalarArray([np.inf, 5] * u.km / u.s / u.pix, axes="channel"),
    dispersion=10 * u.km / u.s / u.pix,
    angle=na.linspace(0, 360, num=36, axis="channel", centers=True) * u.deg,
    wavelength_ref=0 * u.km / u.s,
    position_ref=32 * u.pix,
    coordinates_scene=coordinates_scene,
    coordinates_sensor=coordinates_sensor,
    axis_wavelength="wavelength",
    axis_scene_xy=("scene_x", "scene_y"),
    axis_sensor_xy=("sensor_x", "sensor_y"),
)

In [ ]:
scene = ctis.scenes.gaussians(
    inputs=coordinates_scene,
    width=na.SpectralPositionalVectorArray(30 * u.km / u.s, 1 * u.arcsec),
)

In [ ]:
with astropy.visualization.quantity_support():
    fig, axs = plt.subplots(
        ncols=2,
        gridspec_kw=dict(width_ratios=[.9,.1]),
        constrained_layout=True,
    )
    colorbar = na.plt.rgbmesh(
        C=scene,
        axis_wavelength="wavelength",
        ax=axs[0],
        vmin=0,
        vmax=scene.outputs.max(),
    )
    na.plt.pcolormesh(
        C=colorbar,
        axis_rgb="wavelength",
        ax=axs[1],
    )
    axs[1].yaxis.tick_right()
    axs[1].yaxis.set_label_position("right")

In [ ]:
image = instrument.image(scene)

In [ ]:
with astropy.visualization.quantity_support():
    fig, axs = plt.subplots(
        ncols=2,
        gridspec_kw=dict(width_ratios=[.9,.1]),
        constrained_layout=True,
    )
    ax, cax = axs
    ani, colorbar = na.plt.rgbmovie(
        instrument.angle,
        image.inputs.wavelength,
        image.inputs.position.x,
        image.inputs.position.y,
        C=image.outputs,
        axis_time="channel",
        axis_wavelength="wavelength",
        ax=ax,
        vmin=0,
        vmax=image.outputs.max(),
    )
    na.plt.pcolormesh(
        C=colorbar,
        axis_rgb="wavelength",
        ax=cax,
    )
    ax.set_xlabel(f"detector $x$ ({image.inputs.position.x.unit})")
    ax.set_ylabel(f"detector $y$ ({image.inputs.position.y.unit})")
    cax.xaxis.set_ticks_position("top")
    cax.xaxis.set_label_position("top")
    cax.yaxis.tick_right()
    cax.yaxis.set_label_position("right")

result = ani.to_jshtml(fps=10)
result = IPython.display.HTML(result)

plt.close(ani._fig)

result

In [ ]:
image = image.mean("wavelength")

In [ ]:
backprojected = instrument.backproject(image)

In [ ]:
with astropy.visualization.quantity_support():
    fig, axs = plt.subplots(
        ncols=2,
        gridspec_kw=dict(width_ratios=[.9,.1]),
        constrained_layout=True,
    )
    ax, cax = axs
    ani, colorbar = na.plt.rgbmovie(
        instrument.angle,
        backprojected.inputs.wavelength,
        backprojected.inputs.position.x,
        backprojected.inputs.position.y,
        C=backprojected.outputs,
        axis_time="channel",
        axis_wavelength="wavelength",
        ax=ax,
        vmin=0,
        vmax=backprojected.outputs.max(),
    )
    na.plt.pcolormesh(
        C=colorbar,
        axis_rgb="wavelength",
        ax=cax,
    )
    ax.set_xlabel(f"detector $x$ ({image.inputs.position.x.unit})")
    ax.set_ylabel(f"detector $y$ ({image.inputs.position.y.unit})")
    cax.xaxis.set_ticks_position("top")
    cax.xaxis.set_label_position("top")
    cax.yaxis.tick_right()
    cax.yaxis.set_label_position("right")

result = ani.to_jshtml(fps=10)
result = IPython.display.HTML(result)

plt.close(ani._fig)

result